# 1. Original pipeline: annealed TPS plus MBAR

This notebook runs the method of Dorman et al. on the TinyStories ARI example.
Independent annealed Transition Path Sampling chains populate several bias
windows. MBAR then estimates normalizing constants, overlap, and tail
probabilities at the unbiased state `lambda=0`.

The implementation uses the pinned official supplementary repository and the
local adapter that preserves its cut, suffix-regeneration, and annealing
semantics. This is the original TPS-MBAR family, not our SMC-SNIS estimator.


In [ ]:
# Private-repository setup for Google Colab.
import base64
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

def github_token():
    token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not token:
        try:
            from google.colab import userdata
            token = (userdata.get('GITHUB_TOKEN') or '').strip()
        except Exception:
            token = ''
    if not token:
        raise RuntimeError('Create a Colab secret named GITHUB_TOKEN and enable notebook access.')
    return token

auth = base64.b64encode(f'x-access-token:{github_token()}'.encode()).decode()
def git_cmd(*args):
    return ['git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}', *args]

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(git_cmd('-C', str(REPO_ROOT), 'pull', '--ff-only'), check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(git_cmd('clone', '--recurse-submodules', REPO_URL, str(REPO_ROOT)), check=True)
else:
    subprocess.run(git_cmd('clone', '--recurse-submodules', REPO_URL, str(REPO_ROOT)), check=True)

subprocess.run(git_cmd('-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'), check=True)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
print('Ready:', REPO_ROOT)


In [ ]:
# EDIT PARAMETERS HERE.
RUN_TPS = True
MODEL_NAME = 'roneneldan/TinyStories-8M'
TOKENIZER_NAME = 'EleutherAI/gpt-neo-125M'
PROMPT = 'Once upon a time, in a big forest, there lived a rhinoc'
ARI_CONVENTION = 'official_public'
LAMBDAS = (0.0, -0.5, -1.0, -1.5, -2.0)
TPS_SEEDS = (200, 201, 202)
STEPS_PER_BIAS = 400
BURN_IN_FRACTION = 0.5
NUM_IID_UNBIASED = 1000
MAX_NEW_TOKENS = 100
TAIL_THRESHOLDS = (6.0, 7.0, 8.0)
DEVICE = 'cuda'

assert RUN_TPS, 'Set RUN_TPS=True to generate the in-memory TPS trajectories.'
print('TPS transitions:', len(LAMBDAS) * STEPS_PER_BIAS * len(TPS_SEEDS))


In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from official_annealed_tps_experiment.annealed_tps import (
    OfficialAnnealedTPSConfig,
    run_official_annealed_tps,
)
from official_annealed_tps_experiment.mbar import (
    collect_tps_window_scores,
    estimate_mbar_tails,
)
from rea_llms.models import load_model_and_tokenizer
from rea_llms.observables import create_observable
from rea_llms.generation import prompt_token_ids
from suffix_direct_mh_smc_experiment.smc_sampler import generate_base_completions, score_observable

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
model, tokenizer = load_model_and_tokenizer(
    MODEL_NAME, TOKENIZER_NAME, device=DEVICE, dtype='float32',
)
observable = create_observable(
    'ari', mode='raw', convention=ARI_CONVENTION, cache_enabled=True,
)
print('Model device:', next(model.parameters()).device)


In [ ]:
from tqdm.auto import tqdm

runs = []
for seed in tqdm(TPS_SEEDS, desc='Independent annealed TPS chains'):
    config = OfficialAnnealedTPSConfig(
        max_new_tokens=MAX_NEW_TOKENS,
        lambdas=LAMBDAS,
        steps_per_bias=STEPS_PER_BIAS,
        seed=seed,
        ari_mode='raw',
        tail_thresholds=TAIL_THRESHOLDS,
    )
    runs.append(run_official_annealed_tps(
        model, tokenizer, observable, prompt=PROMPT, config=config,
    ))

run_table = pd.DataFrame([run.summary for run in runs])
display(run_table[[
    'seed', 'runtime_total_seconds', 'acceptance_rate',
    'mean_raw_ari_final_target', 'max_raw_ari_final_target',
    'tps_temporal_ess',
]])


In [ ]:
scores_by_lambda = collect_tps_window_scores(
    runs, burn_in_fraction=BURN_IN_FRACTION,
)
prompt_ids = prompt_token_ids(tokenizer, PROMPT, next(model.parameters()).device)
rng = torch.Generator(device=prompt_ids.device).manual_seed(91001)
iid_tokens = generate_base_completions(
    model, prompt_ids, num_particles=NUM_IID_UNBIASED,
    max_new_tokens=MAX_NEW_TOKENS, batch_size=64, generator=rng,
)
_, iid_ari, _ = score_observable(
    tokenizer, observable, prompt=PROMPT, prompt_ids=prompt_ids, completions=iid_tokens,
)
iid_ari = iid_ari.detach().cpu().numpy()
scores_by_lambda[0.0] = np.concatenate([scores_by_lambda[0.0], iid_ari])
mbar_result = estimate_mbar_tails(
    scores_by_lambda, thresholds=TAIL_THRESHOLDS,
)

normalizers = pd.DataFrame({
    'lambda': mbar_result.lambdas,
    'samples': mbar_result.sample_counts,
    'log Z': mbar_result.log_normalizing_constants,
    'MBAR effective samples': mbar_result.effective_samples,
})
print('MBAR tail estimates under the original model:')
display(mbar_result.tails)
iid_table = pd.DataFrame({
    'threshold': TAIL_THRESHOLDS,
    'IID probability': [np.mean(iid_ari >= value) for value in TAIL_THRESHOLDS],
})
print('Small IID diagnostic (not the high-precision direct reference):')
display(iid_table)
print('Bias-window diagnostics:')
display(normalizers)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].boxplot(
    [scores_by_lambda[value] for value in mbar_result.lambdas],
    tick_labels=[f'{value:g}' for value in mbar_result.lambdas],
    showfliers=False,
)
axes[0].set_xlabel('lambda')
axes[0].set_ylabel('raw ARI')
axes[0].set_title('TPS bias windows')

tail = mbar_result.tails
axes[1].errorbar(
    tail['threshold'], tail['probability'], yerr=1.96 * tail['standard_error'],
    marker='o', capsize=4,
)
axes[1].set_yscale('log')
axes[1].set_xlabel('raw ARI threshold')
axes[1].set_ylabel('base-model probability')
axes[1].set_title('TPS-MBAR tail estimate')

image = axes[2].imshow(mbar_result.overlap_matrix, vmin=0, vmax=1, cmap='viridis')
axes[2].set_xticks(range(len(LAMBDAS)), [f'{x:g}' for x in LAMBDAS], rotation=45)
axes[2].set_yticks(range(len(LAMBDAS)), [f'{x:g}' for x in LAMBDAS])
axes[2].set_title('MBAR overlap matrix')
axes[2].set_xlabel('lambda'); axes[2].set_ylabel('lambda')
fig.colorbar(image, ax=axes[2], fraction=.046)
fig.tight_layout()
plt.show()

print('Caution: TPS samples are correlated. Repeat independent chains and inspect')
print('temporal ESS, between-chain stability, and overlap before using the estimate.')
